<a href="https://colab.research.google.com/github/mic006016/seoul-bigdata-2026/blob/main/seoul_bigdata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder


# ==========================================
# 1. SKT 유동인구 데이터 전처리
# ==========================================
skt = pd.read_csv('/content/SKT 월별 요일별 유동인구.csv', encoding='cp949')

# 1-1) 컬럼명 수정
skt = skt.rename(columns={
    '기준년월(STD_YM)': 'std_ym',
    'X좌표(X_COORD)': 'x_coord',
    'Y좌표(Y_COORD)': 'y_coord',
    '성별코드(GNDR_CD)': 'gndr_cd',
    '연령대코드(AGE_GR_SCTN_CD)': 'age_cd',
    '요일코드(WKDY_CD)': 'wkdy_cd',
    '유동인구수(FLOW_POP_CNT)': 'flow_pop_cnt'
})

# 1-2) 불필요한 컬럼 삭제 (자치구, 시간대코드)
skt = skt.drop(columns=['자치구(SIGUNGU)', '시간대코드(TMST_CD)'], errors='ignore')

# 1-3) 연령대코드 5년 -> 10년 간격으로 통합 함수
def map_skt_age(age):
    if pd.isna(age): return np.nan
    age = int(age)
    if age < 2000: return '1019'       # 0509, 1014, 1519 -> 1019 (20세 미만)
    elif age < 3000: return '2029'     # 2024, 2529 -> 2029
    elif age < 4000: return '3039'
    elif age < 5000: return '4049'
    elif age < 6000: return '5059'
    elif age < 7000: return '6069'
    else: return '7099'                # 70세 이상 통합

skt['age_cd'] = skt['age_cd'].apply(map_skt_age)

# 1-4) 시간대를 삭제했으므로 중복 행을 그룹화하여 유동인구 합산 (메모리 다이어트)
skt_grouped = skt.groupby(
    ['std_ym', 'x_coord', 'y_coord', 'gndr_cd', 'age_cd', 'wkdy_cd']
)['flow_pop_cnt'].sum().reset_index()


# ==========================================
# 2. 신한카드 매출 데이터 전처리
# ==========================================
card = pd.read_csv('/content/3.서울시민의 성별 연령대별(격자별).csv', encoding='cp949')

# 2-1) 컬럼명 수정
card = card.rename(columns={
    '기준일자': 'std_date',
    '격자_250': 'grid_250',
    '개인법인구분': 'pvt_corp',
    '성별': 'gndr_cd',
    '연령대': 'age_cd',
    '업종대분류': 'industry',
    '카드이용금액계': 'sales_amount',
    '카드이용건수계': 'sales_count'
})

# 결측치 제거 (성별, 연령대가 없는 법인카드 등은 분석 목적상 제외하거나 별도 처리 필요)
card = card.dropna(subset=['gndr_cd', 'age_cd'])

# 2-2) 기준년월(std_ym) 컬럼 생성 및 요일코드(wkdy_cd) 추출
# 날짜형식 변환 후 추출 (SKT의 요일코드와 동일하게 1=월요일 ~ 7=일요일로 맞춤)
card['std_date_dt'] = pd.to_datetime(card['std_date'], format='%Y%m%d')
card['std_ym'] = card['std_date'] // 100
card['wkdy_cd'] = card['std_date_dt'].dt.dayofweek + 1

# 2-3) 라벨인코딩 및 매핑
# 성별은 SKT와 맞추기 위해 수동 매핑 (남=1, 여=2)
card['gndr_cd'] = card['gndr_cd'].map({'남': 1, '여': 2})

# 개인법인구분, 업종대분류는 LabelEncoder 사용
le_pvt = LabelEncoder()
le_ind = LabelEncoder()
card['pvt_corp'] = le_pvt.fit_transform(card['pvt_corp'])
card['industry'] = le_ind.fit_transform(card['industry'])

# 나중에 디코딩을 위해 매핑 딕셔너리 출력 저장 추천
print("업종 라벨 매핑:", dict(zip(le_ind.classes_, le_ind.transform(le_ind.classes_))))

# 2-4) 연령대코드(age_cd) 10년 간격 변환
def map_card_age(age):
    age = str(age)
    if age == '20세미만': return '1019'
    elif age == '70세이상': return '7099'
    else: return age[:2] + age[3:5] # '60_69세' -> '60' + '69' -> '6069'

card['age_cd'] = card['age_cd'].apply(map_card_age)

# 2-5) 불필요한 원본 날짜 컬럼 삭제
card_grouped = card.drop(columns=['std_date', 'std_date_dt'])

# 일별 데이터를 묶었으므로 여기서도 동일 기준으로 합산 (메모리 다이어트)
card_grouped = card_grouped.groupby(
    ['std_ym', 'grid_250', 'pvt_corp', 'gndr_cd', 'age_cd', 'industry', 'wkdy_cd']
)[['sales_amount', 'sales_count']].sum().reset_index()


# 결과 확인
print("SKT 전처리 완료 데이터 셋:", skt_grouped.shape)
print("카드 전처리 완료 데이터 셋:", card_grouped.shape)

업종 라벨 매핑: {'ZZ_나머지': np.int64(0), '게임방/오락실': np.int64(1), '결제대행(PG)': np.int64(2), '고속버스/철도/여객선': np.int64(3), '기타식품': np.int64(4), '기타요식': np.int64(5), '기타유통': np.int64(6), '농수산물': np.int64(7), '대형마트': np.int64(8), '면세점': np.int64(9), '생활잡화/수입상품점': np.int64(10), '쇼핑몰': np.int64(11), '수제용품점': np.int64(12), '슈퍼마켓기업형': np.int64(13), '슈퍼마켓일반형': np.int64(14), '스포츠시설': np.int64(15), '시계/귀금속': np.int64(16), '싸우나/목욕탕': np.int64(17), '약국': np.int64(18), '양식': np.int64(19), '영화/공연': np.int64(20), '유흥업소': np.int64(21), '일반병원': np.int64(22), '일식': np.int64(23), '정육점': np.int64(24), '제과점': np.int64(25), '주류판매': np.int64(26), '주유소': np.int64(27), '주차장': np.int64(28), '커피전문점': np.int64(29), '컴퓨터/소프트웨어': np.int64(30), '택시': np.int64(31), '패스트푸드': np.int64(32), '편의점': np.int64(33), '학원/학습지': np.int64(34), '한식': np.int64(35), '할인점/슈퍼마켓/양판점': np.int64(36), '화장품': np.int64(37)}
SKT 전처리 완료 데이터 셋: (500, 7)
카드 전처리 완료 데이터 셋: (181, 9)


In [3]:
skt_grouped.head(5)

,std_ym,x_coord,y_coord,gndr_cd,age_cd,wkdy_cd,flow_pop_cnt
0,201601,950990,1948425,2,1019,7,0.10
1,201601,951332,1947443,1,3039,6,0.18
2,201601,951383,1949804,1,6069,4,0.03
3,201601,951428,1947975,1,7099,4,0.01
4,201601,951531,1948621,1,1019,1,0.19


In [4]:
card_grouped.head(5)

,std_ym,grid_250,pvt_corp,gndr_cd,age_cd,industry,wkdy_cd,sales_amount,sales_count
0,202301,다사38bb52bb,0,1,5059,24,7,325387,21
1,202301,다사39ab51ba,0,1,3039,33,7,491765,59
2,202301,다사39ab53aa,0,2,4049,33,7,82922,32
3,202301,다사40aa42ab,0,1,3039,33,7,190352,43
4,202301,다사40aa42ba,0,2,3039,7,7,120672,16


In [5]:
skt_grouped.to_csv("skt.csv", index=False, encoding="utf-8-sig")
card_grouped.to_csv("card.csv", index=False, encoding="utf-8-sig")

In [7]:
import pandas as pd
from scipy.spatial import cKDTree

# 1. 데이터 로드
match = pd.read_csv('match.csv')
skt = pd.read_csv('skt.csv')
card = pd.read_csv('card.csv')

# =========================================
# [Step 1] K-D Tree 공간 조인 (50m -> 250m)
# =========================================
tree = cKDTree(match[['CELL_X', 'CELL_Y']].values)
distances, indices = tree.query(skt[['x_coord', 'y_coord']].values)
skt['GID'] = match.iloc[indices]['GID'].values

# =========================================
# [Step 2] 시간(std_ym)을 버리고 평균 내기
# =========================================
# SKT: 년월(std_ym)을 빼고 그룹화한 뒤, 유동인구의 '평균(mean)'을 구함.
skt_spatial = skt.groupby(['GID', 'gndr_cd', 'age_cd', 'wkdy_cd'])['flow_pop_cnt'].mean().reset_index()
skt_spatial.rename(columns={'flow_pop_cnt': 'avg_flow_pop'}, inplace=True)

# Card: 카드 데이터도 년월(std_ym)을 빼고 매출액의 '평균(mean)'을 구함.
# (원본에 std_ym이 살아있다면 버리고 그룹화)
card_spatial = card.groupby(['grid_250', 'gndr_cd', 'age_cd', 'wkdy_cd'])[['sales_amount', 'sales_count']].mean().reset_index()
card_spatial.rename(columns={'sales_amount': 'avg_sales', 'sales_count': 'avg_sales_cnt'}, inplace=True)

# =========================================
# [Step 3] 오직 공간(GID)과 고객 속성으로만 결합
# =========================================
final_mart = pd.merge(
    skt_spatial,
    card_spatial,
    left_on=['GID', 'gndr_cd', 'age_cd', 'wkdy_cd'],
    right_on=['grid_250', 'gndr_cd', 'age_cd', 'wkdy_cd'],
    how='inner'
)

# 마무리
final_mart = final_mart.drop(columns=['grid_250'])
final_mart.to_csv('seoul_tourism.csv', index=False, encoding='utf-8-sig')

print(f"최종 데이터 행 수: {len(final_mart)}개")
if len(final_mart) > 0:
    print("\n[데이터 미리보기]")
    print(final_mart.head())


 공간 기반으로 결합된 최종 데이터 행 수: 0개


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# =========================================
# [Step 4] 매크로 타겟팅: 잠재 거점(GID) Top 100 추출
# =========================================
# 성별/연령/요일별로 나뉘어 있는 final_mart를 '동네(GID)' 단위로 모두 결합.
mart_macro = final_mart.groupby('GID')[['avg_flow_pop', 'avg_sales']].sum().reset_index()

# 0~1 사이로 스케일링 (단위가 다른 두 데이터를 공정하게 비교하기 위해)
scaler = MinMaxScaler()
mart_macro[['pop_scaled', 'sales_scaled']] = scaler.fit_transform(mart_macro[['avg_flow_pop', 'avg_sales']])

# 잠재력 점수 = (유동인구 점수 - 매출 점수)
# 사람이 많은데 매출이 안 나오는 곳일수록 점수가 높음.
mart_macro['potential_score'] = mart_macro['pop_scaled'] - mart_macro['sales_scaled']

# 점수 높은 순으로 Top 100 동네(GID) 추출
top_100_gids = mart_macro.sort_values(by='potential_score', ascending=False).head(100)['GID'].tolist()
print(f"Top 100 잠재 거점(GID) 추출 완료")


# =========================================
# [Step 5] 마이크로 타겟팅: 동네별 '핵심 골목' 3곳 추출
# =========================================
# Step 1에서 GID를 부여받은 원본 skt 데이터에서 Top 100 동네에 해당하는 데이터만 필터링
skt_top_100 = skt[skt['GID'].isin(top_100_gids)]

# 각 동네(GID)의 세부 50m 좌표(x, y)별로 유동인구 평균 계산
micro_points = skt_top_100.groupby(['GID', 'x_coord', 'y_coord'])['flow_pop_cnt'].mean().reset_index()

# 각 GID별로 유동인구가 가장 높은 상위 3개 좌표(골목)만 남김
top3_points_per_gid = micro_points.sort_values(['GID', 'flow_pop_cnt'], ascending=[True, False]) \
                                  .groupby('GID').head(3).reset_index(drop=True)
print(f"Top 100 동네 내 핵심 골목(50m) {len(top3_points_per_gid)}곳 추출 완료")


# =========================================
# [Step 6 & 7] 위경도(Lat/Lon) 변환 및 최종 파일 저장")
# =========================================
try:
    from pyproj import Transformer
    has_pyproj = True
except ImportError:
    has_pyproj = False
    print("⚠️ pyproj 모듈이 설치되어 있지 않아 위경도 변환 생략.")

if has_pyproj:
    # 서울시 XY좌표계(EPSG:5181) -> 일반 위경도 좌표계(EPSG:4326) 변환기 생성
    transformer = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)

    def convert_coords(row):
        lon, lat = transformer.transform(row['x_coord'], row['y_coord'])
        return pd.Series({'longitude': lon, 'latitude': lat})

    # 좌표 변환 실행
    latlon_df = top3_points_per_gid.apply(convert_coords, axis=1)
    final_output = pd.concat([top3_points_per_gid, latlon_df], axis=1)
    print("로드뷰용 위경도 변환 완료")
else:
    final_output = top3_points_per_gid


# 최종 반출용 파일 저장
final_output.to_csv('top100_targets.csv', index=False, encoding='utf-8-sig')

print(f"\n 모든 파이프라인 가동 완료")
print(f"'top100_targets.csv'이 저장되었습니다.")
print("\n[최종 반출 데이터 미리보기]")
print(final_output.head())